# Imports:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ase.io import Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor
import seaborn as sns
import matplotlib.pyplot as plt
col_list = sns.color_palette("tab10", 100)

import tqdm

from gnome_aqueous_stability.data_utils import Data_Handler
from gnome_aqueous_stability.analysis_utils import (
    plot_periodic_table_with_values, get_col_dict_for_atoms, 
    Stable_Entries, Stability_Criteria, get_simplified_df, 
    atoms_from_db, Compound_HHI_scores, sys_in_MP_db
)

# pHs = np.linspace(-2, 16, 19) # Data specific
# Us = np.linspace(-2, 4, 31) # Data specific

# Database setup:

In [ ]:
# Define directory of data (can be left as None if the data folder is in 
# the package root):
dir_of_data = None

dh = Data_Handler(
# Whether to apply solid filter:
    solid_filter = True, 
# Whether to use only GGA calculations (True), or include r2SCAN data via the MP-mixing scheme (False):
    gga_only = False,
# Path to data directory:
    path_to_data_directory = dir_of_data
    )

In [ ]:
compounds_df = dh.get_df()
compounds_df.columns

# Inclusion/exclusion of elements:

In [ ]:
radioactive_elements = ['Tc',  'Ra', 'Rf', 'Db', 'Sg', 'Bh', 'Hs',
                        'Mt', 'Ds', 'Rg', 'Cn', 'Nh', 'Fl', 'Mc', 'Lv',
                        'Ts', 'Og', 'Pm', 'Ac', 'Th', 'Pa', 'U', 'Np', 
                        'Pu', 'Am', 'Cm', 'Bk', 'Cf', 'Es', 'Fm', 'Md',
                        'No', 'Lr', 'Po', 'At', 'Rn'
                        ]

reactive_with_water_elements = ['Li', 'Na', 'K', 'Rb', 'Cs', 'Fr', 'Ca', 'Ba']

toxic_elements = ['Tl', 'Pb', 'As', 'Cd', 'Hg']

too_rare_elements = ['Be', 'Y', 'Nb', 'Rh', 'Pd', 'Te', 'Sb', 'Lu', 'Re',
                    'Os', 'Pt', 'Hg', 'Bi', 'La', 'Ce', 'Pr', 'Nd', 'Pm',
                    'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb',
                    'Ru', 'Ir']


CMR = [
    'Be', 'B', 'Si', 'Sc', 'Ti', 'V', 'Co', 'Ga', 'Ge', 'Cr', 'Y', 'Nb', 
    'Ru', 'Rh', 'Pd', 'Sb', 'Lu', 'Hf', 'Ta', 'W', 'Os', 'Ir', 'Pt', 'Bi',
    'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd',
    'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb'
]

platinum_group = ['Ru', 'Rh', 'Pd', 'Os', 'Ir', 'Pt']


elements_to_exclude = radioactive_elements + toxic_elements + reactive_with_water_elements + ['Br', 'C', 'F', 'Se', 'Te', 'B', 'P', 'S', 'Cl'] # + ['S', 'C', 'Cl', 'P', 'B', 'Bi']
 # + higly_concentrated_elements  # elements to discart
elements_whic_must_be_included = ['O']

plot_periodic_table_with_values(
    get_col_dict_for_atoms(elements_to_discart=elements_to_exclude, 
                           elements_to_include=elements_whic_must_be_included)
)

### Discarding materials based on element selection

In [ ]:
dh.restore_df()
dh.remove_entries_with_elements(elements_to_exclude)
dh.remove_entries_without_elements(elements_whic_must_be_included, False)
# dh.remove_entries_without_elements(['Li', 'Na', 'K'], False)

### pbx stability criteria

In [ ]:
SCS = [
    Stability_Criteria(pHs=[0], Us=[1.2, 2], decomposition_threshold=0.5),
    # Stability_Criteria(pHs=[7], Us=[-0.4, -2], decomposition_threshold=0.2),
       # Stability_Criteria(pHs=0, Us=[1.2, 1.6], decomposition_threshold=0.05),
       # Stability_Criteria(pHs=[2, 5], Us=[0., 2], decomposition_threshold=1),
       ]

for sc in SCS:
    sc.visualize()

In [ ]:
se = Stable_Entries(dh, SCS)
se.get_stable_df()

In [ ]:
# df = df[df["Disorder Probability"] < 0.2]
# df = df[~df['Reduced Formula'].apply(lambda x: '(SO4)' in x)]

# df = df[~df['Elements'].apply(lambda x: 'F' in x)]

df = se.get_stable_df().copy()

# df = df[df['Elements'].apply(lambda x: any([el in x for el in ['Cn']]))]
# df = df[df['Elements'].apply(lambda x: any([el in x for el in ['Pd']]))]


df = df[df['Dimensionality Cheon'] == '3D']
# df = df[df['NSites'] < 40]

# df = df[df['Elements'].apply(lambda x: len(x) <= 4)]

df = df[df['Bandgap'] < 0.05]
# df = df[df['average_HHI_P_excluding_OHCNPS'] < 6000]
# df = df[df['average_HHI_R_excluding_OHCNPS'] < 6000]
# # df = df[df['max_HHI_P'] < 6000]
# df = df[df['max_HHI_R'] < 6000]
df = df[df['Disorder Probability'] < 0.2]

# df = get_simplified_df(df)
# df = df.sort_values(by="Bandgap")
# df = df.sort_values(by="HHI_P_excluding_OHCNP")
print(len(df))

df = df.sort_values(by="Bandgap")
get_simplified_df(df)


In [ ]:
simp = sys_in_MP_db('YOUR_MATERIALS_PROJECT_API_KEY')

df = df[~df['Elements'].apply(lambda x: simp.in_db(x))]
print(len(df))
get_simplified_df(df)

In [ ]:
temp_df = df.copy()

temp_df = temp_df[temp_df['Elements'].apply(lambda x: any([el in x for el in ['Pt']]))]
temp_df = temp_df[temp_df['Elements'].apply(lambda x: any([el in x for el in ['Pd']]))]

temp_df

In [ ]:
get_simplified_df(df.sort_values(by="Disorder Probability"))

In [ ]:
retrive_atoms = atoms_from_db()

view(
retrive_atoms.get_atoms_objects_from_df(df.sort_values(by="average_HHI_P_excluding_OHCNPS"))
)

In [ ]:
df.sort_values(by="average_HHI_P_excluding_OHCNPS")